In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "aerolineas_dev")
dbutils.widgets.text("source", "bronze")
dbutils.widgets.text("sink", "silver")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema_source = dbutils.widgets.get("source")
schema_sink = dbutils.widgets.get("sink")

In [0]:
# Tablas de dimensiones
df_bbdd_dim_aerolineas = spark.table(f"{catalog}.{schema_source}.bbdd_dim_aerolineas")
df_bbdd_dim_aeropuertos = spark.table(f"{catalog}.{schema_source}.bbdd_dim_aeropuertos")
df_bbdd_dim_aeronaves = spark.table(f"{catalog}.{schema_source}.bbdd_dim_aeronaves")
df_bbdd_dim_rutas = spark.table(f"{catalog}.{schema_source}.bbdd_dim_rutas")

In [0]:
# Dimensiones Silver
target_table_dim_aerolineas = f"{catalog}.{schema_sink}.tbl_dim_aerolineas"
target_table_dim_aeropuertos = f"{catalog}.{schema_sink}.tbl_dim_aeropuertos"
target_table_dim_aeronaves = f"{catalog}.{schema_sink}.tbl_dim_aeronaves"
target_table_dim_rutas = f"{catalog}.{schema_sink}.tbl_dim_rutas"

## Parametros de limpieza inicial

In [0]:

def validar_dimension(df, condicion_valida, claves, nombre):
    # Mostrar registros que incumplen las reglas
    df_invalidos = df.filter(
        ~F.coalesce(condicion_valida, F.lit(False))
    )

    if df_invalidos.limit(1).count() > 0:
        display(df_invalidos)
        raise ValueError(f"{nombre}: existen registros inválidos.")

    # Validar unicidad de las claves
    for clave in claves:
        duplicados = (
            df.groupBy(clave)
            .count()
            .filter(F.col("count") > 1)
        )

        if duplicados.limit(1).count() > 0:
            display(duplicados)
            raise ValueError(
                f"{nombre}: existen duplicados en {clave}."
            )


def validar_referencia(df, columna, df_referencia, clave, nombre):
    # Detectar claves que no existen en la dimensión de referencia
    referencias = df_referencia.select(
        F.col(clave).alias(columna)
    ).distinct()

    df_sin_referencia = df.join(
        referencias,
        on=columna,
        how="left_anti"
    )

    if df_sin_referencia.limit(1).count() > 0:
        display(df_sin_referencia)
        raise ValueError(
            f"{nombre}: existen referencias inválidas en {columna}."
        )

## Tranformaciones para la tabla tbl_dim_aerolineas
### Limpia textos, normaliza códigos y valida fecha de inicio y estado.

In [0]:
df_dim_aerolineas_silver = df_bbdd_dim_aerolineas.select(
    F.upper(F.trim("aerolinea_id")).alias("aerolinea_id"),
    F.trim("nombre_aerolinea").alias("nombre_aerolinea"),
    F.trim("pais_sede").alias("pais_sede"),
    F.upper(F.trim("hub_iata")).alias("hub_iata"),
    F.upper(F.trim("modelo_negocio")).alias("modelo_negocio"),
    F.expr(
        "try_cast(fecha_inicio_operaciones AS DATE)"
    ).alias("fecha_inicio_operaciones"),
    F.expr("try_cast(activo AS INT)").alias("activo"),
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

condicion_aerolineas = (
    (F.length("aerolinea_id") > 0)
    & (F.length("nombre_aerolinea") > 0)
    & (F.length("pais_sede") > 0)
    & F.col("hub_iata").rlike("^[A-Z]{3}$")
    & (F.length("modelo_negocio") > 0)
    & F.col("fecha_inicio_operaciones").isNotNull()
    & (F.col("fecha_inicio_operaciones") <= F.current_date())
    & F.col("activo").isin(0, 1)
)

validar_dimension(
    df_dim_aerolineas_silver,
    condicion_aerolineas,
    ["aerolinea_id"],
    "dim_aerolineas"
)

target_table = target_table_dim_aerolineas

df_dim_aerolineas_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)

## Tranformaciones para la tabla tbl_dim_aeropuertos
### Normaliza códigos IATA y país, limpia textos y valida coordenadas. La altitud puede ser negativa, por eso no la restringimos a valores positivos.

In [0]:
df_dim_aeropuertos_silver = df_bbdd_dim_aeropuertos.select(
    F.upper(F.trim("aeropuerto_iata")).alias("aeropuerto_iata"),
    F.trim("nombre_aeropuerto").alias("nombre_aeropuerto"),
    F.trim("ciudad").alias("ciudad"),
    F.trim("pais").alias("pais"),
    F.upper(F.trim("codigo_pais")).alias("codigo_pais"),
    F.trim("zona_horaria").alias("zona_horaria"),
    F.expr("try_cast(altitud_m AS INT)").alias("altitud_m"),
    F.expr("try_cast(latitud AS DECIMAL(10,6))").alias("latitud"),
    F.expr("try_cast(longitud AS DECIMAL(10,6))").alias("longitud"),
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

condicion_aeropuertos = (
    F.col("aeropuerto_iata").rlike("^[A-Z]{3}$")
    & (F.length("nombre_aeropuerto") > 0)
    & (F.length("ciudad") > 0)
    & (F.length("pais") > 0)
    & F.col("codigo_pais").rlike("^[A-Z]{2}$")
    & (F.length("zona_horaria") > 0)
    & F.col("altitud_m").isNotNull()
    & F.col("latitud").between(-90, 90)
    & F.col("longitud").between(-180, 180)
)

validar_dimension(
    df_dim_aeropuertos_silver,
    condicion_aeropuertos,
    ["aeropuerto_iata"],
    "dim_aeropuertos"
)

target_table = target_table_dim_aeropuertos

df_dim_aeropuertos_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)

## Tranformaciones para la tabla tbl_dim_aeronaves
### Normaliza identificadores y matrícula, valida capacidad y comprueba que la aerolínea exista.

In [0]:
df_dim_aeronaves_silver = df_bbdd_dim_aeronaves.select(
    F.upper(F.trim("aeronave_id")).alias("aeronave_id"),
    F.upper(F.trim("aerolinea_id")).alias("aerolinea_id"),
    F.upper(F.trim("matricula")).alias("matricula"),
    F.trim("modelo").alias("modelo"),
    F.upper(F.trim("categoria")).alias("categoria"),
    F.expr("try_cast(capacidad_total AS INT)").alias("capacidad_total"),
    F.expr("try_cast(asientos_business AS INT)").alias("asientos_business"),
    F.expr("try_cast(asientos_economy AS INT)").alias("asientos_economy"),
    F.upper(F.trim("tipo_combustible")).alias("tipo_combustible"),
    F.expr("try_cast(anio_fabricacion AS INT)").alias("anio_fabricacion"),
    F.upper(F.trim("estado_aeronave")).alias("estado_aeronave"),
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

condicion_aeronaves = (
    (F.length("aeronave_id") > 0)
    & (F.length("aerolinea_id") > 0)
    & (F.length("matricula") > 0)
    & (F.length("modelo") > 0)
    & (F.length("categoria") > 0)
    & (F.col("capacidad_total") > 0)
    & (F.col("asientos_business") >= 0)
    & (F.col("asientos_economy") >= 0)
    & (
        F.col("asientos_business") + F.col("asientos_economy")
        == F.col("capacidad_total")
    )
    & (F.length("tipo_combustible") > 0)
    & F.col("anio_fabricacion").between(
        1903, F.year(F.current_date())
    )
    & (F.length("estado_aeronave") > 0)
)

validar_dimension(
    df_dim_aeronaves_silver,
    condicion_aeronaves,
    ["aeronave_id", "matricula"],
    "dim_aeronaves"
)

validar_referencia(
    df_dim_aeronaves_silver,
    "aerolinea_id",
    df_dim_aerolineas_silver,
    "aerolinea_id",
    "dim_aeronaves"
)

target_table = target_table_dim_aeronaves

df_dim_aeronaves_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)

## Tranformaciones para la tabla tbl_dim_rutas
### Normaliza códigos, valida distancia y duración, y comprueba que ambos aeropuertos existan.

In [0]:
df_dim_rutas_silver = df_bbdd_dim_rutas.select(
    F.upper(F.trim("ruta_id")).alias("ruta_id"),
    F.upper(F.trim("origen_iata")).alias("origen_iata"),
    F.upper(F.trim("destino_iata")).alias("destino_iata"),
    F.expr("try_cast(distancia_km AS INT)").alias("distancia_km"),
    F.expr(
        "try_cast(duracion_programada_min AS INT)"
    ).alias("duracion_programada_min"),
    F.upper(F.trim("tipo_ruta")).alias("tipo_ruta"),
    F.expr("try_cast(activa AS INT)").alias("activa"),
    "fecha_ingesta",
    "archivo_origen",
    F.current_timestamp().alias("fecha_actualizacion")
)

condicion_rutas = (
    (F.length("ruta_id") > 0)
    & F.col("origen_iata").rlike("^[A-Z]{3}$")
    & F.col("destino_iata").rlike("^[A-Z]{3}$")
    & (F.col("origen_iata") != F.col("destino_iata"))
    & (F.col("distancia_km") > 0)
    & (F.col("duracion_programada_min") > 0)
    & F.col("tipo_ruta").isin("DOMESTICO", "INTERNACIONAL")
    & F.col("activa").isin(0, 1)
)

validar_dimension(
    df_dim_rutas_silver,
    condicion_rutas,
    ["ruta_id"],
    "dim_rutas"
)

validar_referencia(
    df_dim_rutas_silver,
    "origen_iata",
    df_dim_aeropuertos_silver,
    "aeropuerto_iata",
    "dim_rutas"
)

validar_referencia(
    df_dim_rutas_silver,
    "destino_iata",
    df_dim_aeropuertos_silver,
    "aeropuerto_iata",
    "dim_rutas"
)

target_table = target_table_dim_rutas

df_dim_rutas_silver.write \
    .mode("overwrite") \
    .insertInto(target_table)